<a href="https://colab.research.google.com/github/kaushiksrj17-cmyk/CodeAlpha_Tasks/blob/main/Task3_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch Version :", torch.__version__)
print("GPU Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))


In [ ]:
!pip -q install ultralytics
!pip -q install supervision
!pip -q install opencv-python-headless
!pip -q install pandas
!pip -q install numpy
!pip -q install matplotlib
!pip -q install pillow
!pip -q install lap

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
import supervision as sv
from google.colab import files
from datetime import datetime
import os

In [ ]:
folders = [
    "models",
    "assets",
    "output",
    "screenshots",
    "logs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

print("\nProject Structure")

for folder in folders:
    print("📁", folder)


In [ ]:
model = YOLO("yolov8n.pt")

print("YOLOv8 Nano Model Loaded Successfully!")

In [ ]:
print(model.names)

In [ ]:
from google.colab import files

In [ ]:
uploaded = files.upload()

In [ ]:
video_name = list(uploaded.keys())[0]

print("Uploaded Video:", video_name)

In [ ]:
video_path = list(uploaded.keys())[0]

print("Video Loaded:", video_path)

In [ ]:
!pip install -q ultralytics supervision lap pandas opencv-python-headless

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
print("YOLO Loaded Successfully!")

In [ ]:
import cv2
import numpy as np
import os
import time
from ultralytics import YOLO

In [ ]:
from google.colab import files

uploaded = files.upload()

video_path = list(uploaded.keys())[0]

In [ ]:
os.makedirs("output", exist_ok=True)

print("Output folder created successfully!")

In [ ]:
cap = cv2.VideoCapture(video_path)

if cap.isOpened():
    print("Video opened successfully!")
else:
    print("Failed to open video.")

In [ ]:
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

print(f"Width : {width}")
print(f"Height: {height}")
print(f"FPS   : {fps}")

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    "output/detection_output.mp4",
    fourcc,
    fps,
    (width, height)
)

print("Output video initialized!")

In [ ]:
import random

def get_color(class_id):
    random.seed(class_id)

    return (
        random.randint(60,255),
        random.randint(60,255),
        random.randint(60,255)
    )

In [ ]:
frame_count = 0

start_time = time.time()

while True:

    success, frame = cap.read()

    if not success:
        break

    frame_count += 1

    results = model(frame, verbose=False)

    annotated = frame.copy()

    for result in results:

        for box in result.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            confidence = float(box.conf[0])

            class_id = int(box.cls[0])

            class_name = model.names[class_id]

            color = get_color(class_id)

            # Bounding Box
            cv2.rectangle(
                annotated,
                (x1, y1),
                (x2, y2),
                color,
                3
            )

            # Label
            label = f"{class_name} {confidence:.2f}"

            cv2.putText(
                annotated,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2
            )

    out.write(annotated)

In [ ]:
cap.release()
out.release()

print("Object Detection Completed Successfully!")

In [ ]:
from google.colab import files

files.download("output/detection_output.mp4")

In [ ]:
tracker_config = """
tracker_type: bytetrack
track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.6
track_buffer: 30
match_thresh: 0.8
fuse_score: True
"""

with open("bytetrack.yaml", "w") as f:
    f.write(tracker_config)

print("ByteTrack configuration created successfully!")

In [ ]:
import cv2
import numpy as np
import pandas as pd
import random
import os
import time
from datetime import datetime
from collections import defaultdict
from ultralytics import YOLO

In [ ]:
model = YOLO("yolov8n.pt")

In [ ]:
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

os.makedirs("output", exist_ok=True)

writer = cv2.VideoWriter(
    "output/tracked_output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

In [ ]:
object_counter = defaultdict(int)

track_history = {}

frame_number = 0

start_time = time.time()

In [ ]:
def get_color(track_id):
    random.seed(track_id)

    return (
        random.randint(50,255),
        random.randint(50,255),
        random.randint(50,255)
    )

In [ ]:
# ===========================
# Professional Color Function
# ===========================

def get_color(track_id):
    track_id = int(track_id)

    return (
        (track_id * 37) % 255,
        (track_id * 17) % 255,
        (track_id * 97) % 255
    )

In [ ]:
# ==========================================
# AI Professional Tracking Engine
# ==========================================

while True:

    success, frame = cap.read()

    if not success:
        break

    frame_number += 1

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )

    annotated = frame.copy()

    elapsed = time.time() - start_time
    fps_live = frame_number / elapsed if elapsed > 0 else 0

    current_objects = 0

    if len(results) > 0:

        result = results[0]

        if result.boxes.id is not None:

            boxes = result.boxes.xyxy.cpu().numpy()
            ids = result.boxes.id.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy().astype(int)
            confs = result.boxes.conf.cpu().numpy()

            for box, track_id, cls, conf in zip(boxes, ids, classes, confs):

                track_id = int(track_id)

                x1, y1, x2, y2 = map(int, box)

                class_name = model.names[cls]

                current_objects += 1

                object_counter[class_name] += 1

                color = get_color(track_id)

                # Bounding Box
                cv2.rectangle(
                    annotated,
                    (x1, y1),
                    (x2, y2),
                    color,
                    3
                )

                # Label
                label = f"{class_name} | ID:{track_id} | {conf:.2f}"

                cv2.putText(
                    annotated,
                    label,
                    (x1, max(y1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2
                )

    # ===================================
    # Dashboard
    # ===================================

    cv2.rectangle(
        annotated,
        (10, 10),
        (370, 180),
        (35, 35, 35),
        -1
    )

    cv2.putText(
        annotated,
        "AI OBJECT DETECTOR & TRACKER",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        annotated,
        f"FPS : {fps_live:.1f}",
        (20, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    cv2.putText(
        annotated,
        f"Objects : {current_objects}",
        (20, 95),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    cv2.putText(
        annotated,
        f"Frame : {frame_number}",
        (20, 125),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    cv2.putText(
        annotated,
        datetime.now().strftime("%d-%m-%Y %H:%M:%S"),
        (20, 155),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 0),
        2
    )

    writer.write(annotated)

print("Tracking Completed Successfully!")

In [ ]:
cap.release()
writer.release()

print("Output video saved successfully!")

In [ ]:
from google.colab import files

files.download("output/tracked_output.mp4")

In [ ]:
from collections import defaultdict
import pandas as pd

# Store detection data
detection_log = []

# Count unique objects
unique_tracks = set()

# Count objects by class
class_counter = defaultdict(int)

# Store movement history
track_history = defaultdict(list)

print("Analytics initialized successfully!")

In [ ]:
track_id = int(track_id)

x1, y1, x2, y2 = map(int, box)

class_name = model.names[cls]

# Count only new tracked objects
if track_id not in unique_tracks:
    unique_tracks.add(track_id)
    class_counter[class_name] += 1

# Save detection log
detection_log.append({
    "Frame": frame_number,
    "Track_ID": track_id,
    "Class": class_name,
    "Confidence": round(float(conf), 2),
    "Time": datetime.now().strftime("%H:%M:%S")
})

# Motion trail
center_x = (x1 + x2) // 2
center_y = (y1 + y2) // 2

track_history[track_id].append((center_x, center_y))

if len(track_history[track_id]) > 30:
    track_history[track_id].pop(0)

# Draw trail
points = track_history[track_id]

for i in range(1, len(points)):
    cv2.line(
        annotated,
        points[i-1],
        points[i],
        get_color(track_id),
        2
    )

current_objects += 1

color = get_color(track_id)

cv2.rectangle(
    annotated,
    (x1, y1),
    (x2, y2),
    color,
    3
)

label = f"{class_name} ID:{track_id} {conf:.2f}"

cv2.putText(
    annotated,
    label,
    (x1, max(20, y1-10)),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.6,
    color,
    2
)

In [ ]:
cv2.rectangle(annotated, (10,10), (400,250), (30,30,30), -1)

cv2.putText(
    annotated,
    "AI OBJECT DETECTOR & TRACKER",
    (20,35),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.7,
    (0,255,255),
    2
)

cv2.putText(
    annotated,
    f"FPS : {fps_live:.1f}",
    (20,65),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.6,
    (255,255,255),
    2
)

cv2.putText(
    annotated,
    f"Objects in Frame : {current_objects}",
    (20,95),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.6,
    (255,255,255),
    2
)

cv2.putText(
    annotated,
    f"Unique Objects : {len(unique_tracks)}",
    (20,125),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.6,
    (255,255,255),
    2
)

y = 160

for name, count in class_counter.items():

    cv2.putText(
        annotated,
        f"{name}: {count}",
        (20,y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0,255,0),
        2
    )

    y += 25

In [ ]:
writer.release()

In [ ]:
df = pd.DataFrame(detection_log)

df.to_csv("output/detections.csv", index=False)

print(df.head())

In [ ]:
from google.colab import files

files.download("output/tracked_output.mp4")
files.download("output/detections.csv")

In [ ]:
import os

os.makedirs("screenshots", exist_ok=True)

In [ ]:
writer.write(annotated)

In [ ]:
# Save a screenshot every 100 frames

if frame_number % 100 == 0:

    filename = f"screenshots/frame_{frame_number}.jpg"

    cv2.imwrite(filename, annotated)

In [ ]:
cv2.putText(
    annotated,
    "Developed by S. Kaushik",
    (width-320, height-20),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.6,
    (255,255,255),
    2
)

In [ ]:
writer.release()

In [ ]:
total_frames = frame_number

total_unique_objects = len(unique_tracks)

processing_time = time.time() - start_time

average_fps = total_frames / processing_time

print("="*50)

print("PROJECT SUMMARY")

print("="*50)

print(f"Frames Processed : {total_frames}")

print(f"Unique Objects  : {total_unique_objects}")

print(f"Processing Time : {processing_time:.2f} sec")

print(f"Average FPS     : {average_fps:.2f}")

print("="*50)

In [ ]:
report = pd.DataFrame({

    "Metric":[
        "Frames Processed",
        "Unique Objects",
        "Processing Time",
        "Average FPS"
    ],

    "Value":[
        total_frames,
        total_unique_objects,
        processing_time,
        average_fps
    ]

})

report.to_csv("output/project_report.csv",index=False)

print(report)

In [ ]:
from google.colab import files

files.download("output/tracked_output.mp4")

files.download("output/detections.csv")

files.download("output/project_report.csv")

In [ ]:
import os

print(os.listdir("output"))

In [ ]:
from IPython.display import Video

Video("output/tracked_output.mp4", embed=True)

In [ ]:
import pandas as pd

df = pd.read_csv("output/detections.csv")

df.head()

In [ ]:
report = pd.read_csv("output/project_report.csv")

report

In [ ]:
import os
os.listdir("output")

In [ ]:
import pandas as pd
pd.read_csv("output/detections.csv").head()

In [ ]:
from IPython.display import Video

Video("output/tracked_output.mp4", embed=True)